# Lesson 29: Regularization

In Lesson 28 we chose a model's complexity by comparing models with cross-validation: polynomial degrees, and then every subset of 7 features for the 1985 car prices. With many features, trying every subset is impossible (50 features would be about $10^{15}$ models). **Regularization** keeps every feature but limits the size of the coefficients, and one hyperparameter, $\alpha$, controls how much. We choose $\alpha$ with cross-validation.

This notebook follows the **Regularization Explorer** (`reg-explorer/index.html`) section by section, with the same data and the same numbers:

1. **A little of each:** 12 cars, a degree-6 polynomial, and shrinking every $\theta$
2. **Correlated features:** why least squares picks huge, unstable $\theta$s
3. **LASSO (L1)**
4. **Ridge (L2)**
5. **Choosing $\alpha$ with cross-validation** (the 1985 price data)

At the end, an **optional practice** section applies the same tools to a market-analysis survey with 35 features.

[Acknowledgments Page](https://ds100.org/fa23/acks/)

In [ ]:
# Run this cell to set up your notebook
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
sns.set()
sns.set_context("talk")

import plotly.express as px

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn import model_selection

def mse(y, yhat):
    return np.mean((np.asarray(y) - np.asarray(yhat))**2)

def rmse(y, yhat):
    return np.sqrt(mse(y, yhat))

### Our objective, and sklearn's

On the slides (and in the explorer) LASSO and ridge minimize

$$
\underbrace{\frac{1}{n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2}_{\text{MSE}} ~+~ \alpha \sum_{j = 1}^{p} |\theta_j| \qquad\text{and}\qquad \frac{1}{n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 ~+~ \alpha \sum_{j = 1}^{p} \theta_j^2 .
$$

`sklearn` writes the same problems with the error part scaled differently: `Lasso` minimizes $\frac{1}{2n}\sum (y_i - \hat{y}_i)^2 + \alpha_{\text{sk}} \sum |\theta_j|$, and `Ridge` minimizes $\sum (y_i - \hat{y}_i)^2 + \alpha_{\text{sk}} \sum \theta_j^2$. Same answers, as long as we convert:

* LASSO: $\alpha_{\text{sk}} = \alpha / 2$
* Ridge: $\alpha_{\text{sk}} = n\,\alpha$ ($n$ = number of training rows)

The two helper functions below do that conversion, so the $\alpha$ values in this notebook match the slides and the explorer. Both standardize the features first (with the training rows' means and SDs) and leave the intercept unpenalized.

In [ ]:
def lasso_model(alpha):
    """LASSO with the slides' scaling of alpha, on standardized features."""
    return make_pipeline(StandardScaler(), Lasso(alpha=alpha / 2, max_iter=100_000, tol=1e-10))

def ridge_model(alpha, n):
    """Ridge with the slides' scaling of alpha (n = number of training rows), on standardized features."""
    return make_pipeline(StandardScaler(), Ridge(alpha=n * alpha))

def ols_model():
    return make_pipeline(StandardScaler(), LinearRegression())

def coefs(model):
    """The theta_1, ..., theta_p of a fitted pipeline (for the standardized features)."""
    return model[-1].coef_

# 1. A little of each

Like Lesson 28, we train on a small sample: the **same 12 cars** the explorer uses. The other 380 cars are new cars the model has never seen.

In [ ]:
vehicles = sns.load_dataset("mpg").dropna().reset_index(drop=True).rename(columns={"horsepower": "hp"})

# the explorer's 12 training cars (its SEED = 126); every other car is a "new car"
train_rows = [20, 203, 7, 183, 211, 178, 124, 236, 29, 53, 38, 67]
train = vehicles.loc[train_rows].sort_values("hp")
new = vehicles.drop(index=train_rows)
print(len(train), "training cars,", len(new), "new cars")
train[["name", "hp", "mpg"]]

We fit a **degree-6 polynomial**: $\hat{y} = \theta_0 + \theta_1\phi_1 + \dots + \theta_6\phi_6$, where $\phi_k$ is $\text{hp}^k$ standardized with the 12 cars' mean and SD. (We divide hp by 100 before taking powers; that only helps the arithmetic, and after standardizing it changes nothing.)

In [ ]:
def powers(df, degree):
    hp = df["hp"].to_numpy() / 100
    return pd.DataFrame({f"hp^{k}": hp**k for k in range(1, degree + 1)})

X6_train, X6_new = powers(train, 6), powers(new, 6)
y_train, y_new = train["mpg"].to_numpy(), new["mpg"].to_numpy()

ols6 = ols_model().fit(X6_train, y_train)
print("training MSE:", round(mse(y_train, ols6.predict(X6_train)), 1))
print("new-car MSE: ", round(mse(y_new, ols6.predict(X6_new)), 1))
pd.Series(coefs(ols6), index=[f"theta_{k}" for k in range(1, 7)]).round(0)

Training MSE 7.2, but 88.6 on new cars: **overfitting**. And look at the coefficients: enormous, alternating in sign, and nearly cancelling each other out.

Let's draw the fit (`hp_grid` is a fine grid of horsepowers, used for drawing curves).

In [ ]:
hp_grid = pd.DataFrame({"hp": np.linspace(46, 230, 300)})

def plot_fits(models, labels, ylim=(-10, 50)):
    plt.figure(figsize=(9, 5))
    plt.scatter(new["hp"], new["mpg"], s=8, color="lightgray", label="380 new cars")
    plt.scatter(train["hp"], train["mpg"], s=40, color="black", label="12 training cars")
    for m, lab in zip(models, labels):
        plt.plot(hp_grid["hp"], m(hp_grid), lw=2.5, label=lab)
    plt.ylim(*ylim); plt.xlabel("horsepower"); plt.ylabel("mpg"); plt.legend(fontsize=11)

plot_fits([lambda d: ols6.predict(powers(d, 6))], ["degree 6, least squares"])

### A first idea: shrink every $\theta$

Use each feature only "a little": multiply $\theta_1, \dots, \theta_6$ by a **shrink factor** $s$ between 0 and 1, and keep $\theta_0$.

In [ ]:
theta0, theta = ols6[-1].intercept_, coefs(ols6)
Z_train = ols6[0].transform(X6_train)     # standardized with the 12 cars' means and SDs
Z_new = ols6[0].transform(X6_new)

def shrunk_predict(Z, s):
    return theta0 + Z @ (s * theta)

rows = []
for s in [1, 0.5, 0.4, 0.2, 0]:
    rows.append({"s": s, "training MSE": mse(y_train, shrunk_predict(Z_train, s)),
                 "new-car MSE": mse(y_new, shrunk_predict(Z_new, s))})
pd.DataFrame(rows).round(1)

Shrinking trades training error for new-car error, the trade-off from last time. At $s = 0$ every feature is switched off and the model is a flat line at $\theta_0$, the average mpg of the 12 cars:

In [ ]:
print("theta_0 =", round(theta0, 2), "  average mpg of the 12 cars =", round(y_train.mean(), 2))

**Why not shrink $\theta_0$ too?** Then $s = 0$ would predict 0 mpg for every car. So the intercept is never penalized.

Shrinking every $\theta$ by the same factor is crude: its best new-car MSE is about 40. Better: let the data decide which $\theta$s to shrink, and by how much, by adding a **penalty** to the error:

$$\text{minimize} \quad \text{MSE}(\theta) + \text{penalty}(\theta)$$

But first: why did least squares pick such enormous $\theta$s?

# 2. Correlated features: why least squares goes wild

Start with just hp and hp², a quadratic. Among the 12 cars these two columns are almost the same:

In [ ]:
X2_train, X2_new = powers(train, 2), powers(new, 2)
print("correlation of hp and hp^2:", round(np.corrcoef(X2_train["hp^1"], X2_train["hp^2"])[0, 1], 3))

ols2 = ols_model().fit(X2_train, y_train)
print("theta_1, theta_2 =", coefs(ols2).round(1), "  training MSE:", round(mse(y_train, ols2.predict(X2_train)), 2))

Because the columns carry almost the same information, we can push $\theta_1$ up and $\theta_2$ down together and the model barely changes. Here we move along that direction and recompute the training MSE (the explorer's "swing"):

In [ ]:
Z2 = ols2[0].transform(X2_train)
r = np.corrcoef(X2_train["hp^1"], X2_train["hp^2"])[0, 1]
for t in [-10, -5, 0, 5, 10]:
    th = coefs(ols2) + t * np.array([1, -r])
    print(f"theta = ({th[0]:6.1f}, {th[1]:6.1f})   training MSE = {mse(y_train, ols2[-1].intercept_ + Z2 @ th):.2f}")

Many very different pairs of $\theta$s give nearly the same training MSE. The data can't tell them apart.

### All six powers

With all six powers, neighbouring columns are even more alike:

In [ ]:
X6_train.corr().round(3)

**Why are the six $\theta$s so big?** Standardizing puts every column on the same scale, but it doesn't change the correlations. The columns are almost identical, so the only way to tell them apart is through their tiny differences. For example, $\phi_1 - \phi_2$ has an SD of only about 0.14:

In [ ]:
print("SD of phi_1 - phi_2:", round(np.std(Z_train[:, 0] - Z_train[:, 1]), 3))

# the smallest mixes of the six columns: eigenvalues of the correlation matrix
eigvals, eigvecs = np.linalg.eigh(np.corrcoef(Z_train, rowvar=False))
print("SD of each of the 6 'mixes' of the columns:", np.sqrt(np.abs(eigvals)).round(6))

The smallest mix of the six columns has an SD of about 0.000005. To bend the curve through the last few training cars, least squares multiplies those tiny differences by huge numbers. Smaller $\theta$s that still cancel exist (we'll see one below), but they fit the 12 cars slightly worse, and least squares only cares about training error.

### The valley

Move all six $\theta$s together along the flattest direction (the smallest eigenvector), by thousands. Between the 12 cars the curve barely moves:

In [ ]:
v = eigvecs[:, 0] / np.max(np.abs(eigvecs[:, 0]))      # the flattest direction, biggest entry = 1
for t in [-10_000, 0, 10_000]:
    th = theta + t * v
    pred = theta0 + Z_train @ th
    print(f"move {t:>7,}: theta = {np.round(th).astype(int)}   training MSE = {mse(y_train, pred):.2f}")

### New samples, new $\theta$s

Draw a new sample of 12 cars and refit. The $\theta$s jump to completely different huge values: this is the **variance** from last time, seen in parameter space. (These samples use numpy's random numbers, so they're not the explorer's samples, but they behave the same way.)

In [ ]:
rng = np.random.default_rng(29)
for k in range(4):
    smp = vehicles.loc[rng.choice(len(vehicles), size=12, replace=False)]
    m = ols_model().fit(powers(smp, 6), smp["mpg"])
    print(f"sample {k + 1}: biggest |theta| = {np.max(np.abs(coefs(m))):>12,.0f}")

### The fix: limit the $\theta$s

Don't let the $\theta$s wander far from 0: set a limit and pick the best model that respects it. Here is one such model, ridge regression with $\alpha = 0.001$ (Section 4 explains ridge):

In [ ]:
ridge6 = ridge_model(0.001, n=12).fit(X6_train, y_train)
print("theta:", coefs(ridge6).round(2))
print("training MSE:", round(mse(y_train, ridge6.predict(X6_train)), 2), "(least squares: 7.22)")
print("new-car MSE: ", round(mse(y_new, ridge6.predict(X6_new)), 1), "(least squares: 88.6)")
plot_fits([lambda d: ols6.predict(powers(d, 6)), lambda d: ridge6.predict(powers(d, 6))],
          ["least squares", "ridge, alpha = 0.001"])

Every $\theta$ is under 15, the dive is gone, and the new-car MSE drops from 88.6 to 20.5, while the training MSE only rises from 7.22 to 9.31. How do we set the limit? Two ways to measure the size of the $\theta$s: $\sum|\theta_j|$ (L1, LASSO) and $\sum\theta_j^2$ (L2, ridge).

# 3. L1 regularization (LASSO)

Back to two features, hp and hp². Every point $(\theta_1, \theta_2)$ is one model. The contours below connect models with the same training MSE; the lowest is at the least-squares point. (Two of the contours, MSE 11.84 and 17.85, are the ones that touch the budgets we use below.)

In [ ]:
t1, t2 = np.meshgrid(np.linspace(-40, 40, 241), np.linspace(-40, 40, 241))
b0 = ols2[-1].intercept_
mse_grid = np.mean((y_train[:, None, None] - (b0 + Z2[:, 0, None, None] * t1 + Z2[:, 1, None, None] * t2))**2, axis=0)

def plane(points=(), shapes=(), lim=40):
    plt.figure(figsize=(6, 6))
    plt.contour(t1, t2, mse_grid, levels=[9.5, 11.84, 15, 17.85, 25, 35, 50, 70], cmap="Blues_r")
    plt.axhline(0, color="k", lw=1); plt.axvline(0, color="k", lw=1)
    plt.plot(*coefs(ols2), "ko"); plt.annotate("least squares", coefs(ols2), xytext=(5, 5), textcoords="offset points", fontsize=11)
    for q, kind in shapes:
        u = np.linspace(0, 2 * np.pi, 400)
        if kind == "L1": xs, ys = q * np.sign(np.cos(u)) * np.cos(u)**2, q * np.sign(np.sin(u)) * np.sin(u)**2
        else: xs, ys = np.sqrt(q) * np.cos(u), np.sqrt(q) * np.sin(u)
        plt.fill(xs, ys, alpha=0.3, color="green" if kind == "L1" else "orange")
    for th, lab in points:
        plt.plot(*th, "o", color="red"); plt.annotate(lab, th, xytext=(5, -15), textcoords="offset points", fontsize=11)
    plt.xlabel(r"$\theta_1$ (hp)"); plt.ylabel(r"$\theta_2$ (hp$^2$)"); plt.gca().set_aspect("equal")
    plt.xlim(-lim, lim); plt.ylim(-lim, lim)

plane()

The long, thin valley of the contours is the correlation again. Now give the model a **budget**: $|\theta_1| + |\theta_2| \le Q$ (the intercept isn't included). The allowed models fill a diamond, and we pick the allowed model with the lowest MSE.

The penalty form gives the same answers: for every $Q$ there is an $\alpha$ with the same $\theta$s. The explorer's readout showed $Q = 20 \leftrightarrow \alpha \approx 0.23$ and $Q = 5 \leftrightarrow \alpha \approx 2.8$. Let's check with `Lasso`:

In [ ]:
for a in [0.23, 2.8]:
    m = lasso_model(a).fit(X2_train, y_train)
    th = coefs(m)
    print(f"alpha = {a}: theta = ({th[0]:.1f}, {th[1]:.1f}),  |theta_1| + |theta_2| = {np.abs(th).sum():.1f}")

In [ ]:
plane(points=[(coefs(lasso_model(0.23).fit(X2_train, y_train)), "Q = 20"),
              (coefs(lasso_model(2.8).fit(X2_train, y_train)), "Q = 5")],
      shapes=[(20, "L1"), (5, "L1")])

At $Q = 5$ the contours touch the diamond at a **corner**, where $\theta_2 = 0$ exactly: hp² is dropped from the model. Corners stick out, so the contours often hit them first. That's why LASSO sets some $\theta$s exactly to 0: it does **feature selection**. (LASSO = "least absolute shrinkage and selection operator".)

As $\alpha$ grows the $\theta$s shrink, and eventually both are 0: the flat line at $\theta_0$.

In [ ]:
for a in [0, 0.1, 0.23, 1, 2.8, 10, 100]:
    m = ols_model() if a == 0 else lasso_model(a)
    m.fit(X2_train, y_train)
    print(f"alpha = {a:>5}: theta = {coefs(m).round(2)},  training MSE = {mse(y_train, m.predict(X2_train)):6.2f},  new-car MSE = {mse(y_new, m.predict(X2_new)):6.2f}")

LASSO has no closed-form solution, so `sklearn` finds the $\theta$s numerically.

# 4. L2 regularization (ridge)

A different budget: $\theta_1^2 + \theta_2^2 \le Q$, a circle. A circle has no corners, so both $\theta$s shrink and neither lands exactly on 0. Compare LASSO and ridge at the **same size**, both crossing the axes at $\pm 5$ (the explorer's step 2):

In [ ]:
def ridge_with_budget(Q, X, y, lo=1e-6, hi=1e4):
    """Find the ridge alpha whose theta_1^2 + theta_2^2 equals Q (bisection on a log scale)."""
    for _ in range(100):
        mid = np.sqrt(lo * hi)
        if np.sum(coefs(ridge_model(mid, len(y)).fit(X, y))**2) > Q: lo = mid
        else: hi = mid
    return mid

a_l2 = ridge_with_budget(25, X2_train, y_train)
th_l1 = coefs(lasso_model(2.8).fit(X2_train, y_train))
th_l2 = coefs(ridge_model(a_l2, 12).fit(X2_train, y_train))
print("L1 (diamond, Q = 5): theta =", th_l1.round(2))
print(f"L2 (circle, Q = 25): theta = {th_l2.round(2)}   (alpha = {a_l2:.3f})")
plane(points=[(th_l1, "L1"), (th_l2, "L2")], shapes=[(5, "L1"), (25, "L2")], lim=12)

L1 selects; L2 shrinks everything.

### The closed form

Unlike LASSO, ridge regression has a closed-form solution. With standardized features $\mathbb{Z}$ and centered $\mathbb{Y}$ (that's how the intercept stays unpenalized):

$$\hat{\theta}_{\text{ridge}} = (\mathbb{Z}^T\mathbb{Z} + n\alpha I)^{-1}\mathbb{Z}^T\mathbb{Y}$$

Let's check it against `sklearn` on the six-power model from Section 2:

In [ ]:
n, alpha = 12, 0.001
Z = StandardScaler().fit_transform(X6_train)
Yc = y_train - y_train.mean()
theta_closed = np.linalg.solve(Z.T @ Z + n * alpha * np.eye(6), Z.T @ Yc)
print("closed form:", theta_closed.round(3))
print("sklearn:    ", coefs(ridge6).round(3))

$\mathbb{Z}^T\mathbb{Z} + n\alpha I$ is invertible for any $\alpha > 0$, even when $\mathbb{Z}^T\mathbb{Z}$ isn't (for example, with perfectly collinear features). That's one reason ridge is popular.

*Elastic net* uses both penalties at once ($\alpha_1\sum|\theta_j| + \alpha_2\sum\theta_j^2$); it's beyond this course (`sklearn`: `ElasticNet`).

# 5. Choosing $\alpha$ with cross-validation

Back to last time's cliffhanger: 199 cars from 1985, predicting **price** from 7 engineered features of horsepower and city mpg. The `cv_fold` column holds the explorer's 5 CV folds (the same folds as Lesson 28), so our numbers match the explorer.

*No test set here, on purpose:* with only 199 cars, a 40-car test set would be too small to tell these models apart. In practice you'd lock one away and use it once, as in Lesson 28.

In [ ]:
cars85 = pd.read_csv("data/auto85_price.csv")
hp, m = cars85["horsepower"], cars85["city_mpg"]
X85 = pd.DataFrame({"hp": hp, "hp^2": hp**2, "log hp": np.log(hp),
                    "mpg": m, "mpg^2": m**2, "1/mpg": 1 / m, "hp*mpg": hp * m})
y85 = cars85["price"] / 1000          # price in $1000s, as in the explorer
folds = cars85["cv_fold"]
X85.corr().round(2)

Most of the features are strongly correlated, just like the powers of hp. (We measure price in \$1000s, as the explorer does; the size of $y$ affects which $\alpha$ is best, so the units matter when you compare $\alpha$ values.) Here is 5-fold CV as a function: for each fold, fit on the other four folds (the pipeline standardizes with *those* rows' means and SDs, never the validation rows') and score on the held-out fold.

In [ ]:
def cv_rmse(make_model, X, y, folds):
    """5-fold CV RMSE. make_model(n_train) returns an unfitted model."""
    sq_err = 0
    for k in sorted(folds.unique()):
        tr, va = folds != k, folds == k
        model = make_model(tr.sum()).fit(X[tr], y[tr])
        sq_err += np.sum((y[va] - model.predict(X[va]))**2)
    return np.sqrt(sq_err / len(y))

print(f"least squares, all 7 features: CV RMSE ${1000 * cv_rmse(lambda n: ols_model(), X85, y85, folds):,.0f}")

Last time, CV on all 128 subsets picked 4 features. Let's reproduce that with the same function (this takes a few seconds):

In [ ]:
from itertools import combinations

subset_cv = {cols: cv_rmse(lambda n: ols_model(), X85[list(cols)], y85, folds)
             for k in range(1, 8) for cols in combinations(X85.columns, k)}
best_subset = min(subset_cv, key=subset_cv.get)
print(f"best of {len(subset_cv) + 1} subsets: {best_subset}, CV RMSE ${1000 * subset_cv[best_subset]:,.0f}")

### LASSO: one knob instead of 128 models

Keep all 7 features and fit LASSO for 121 values of $\alpha$ from $10^{-4}$ to $10^{2}$. First the coefficient paths (the explorer's "parameter star, unrolled"):

In [ ]:
alphas = np.logspace(-4, 2, 121)
paths = pd.DataFrame([coefs(lasso_model(a).fit(X85, y85)) for a in alphas], columns=X85.columns, index=alphas)

paths.plot(logx=True, figsize=(9, 5), lw=2)
plt.axhline(0, color="k", lw=1)
plt.xlabel(r"$\alpha$ (log scale)"); plt.ylabel(r"$\theta$ (\$1000 per SD)"); plt.legend(fontsize=10, ncol=2);

In [ ]:
print("number of nonzero coefficients at selected alphas:")
(paths.abs() > 1e-9).sum(axis=1).iloc[::20]

The $\theta$s shrink and drop to exactly 0 one after another. Now run 5-fold CV for every $\alpha$, the same recipe as for the polynomial degree:

In [ ]:
lasso_cv = pd.Series([cv_rmse(lambda n: lasso_model(a), X85, y85, folds) for a in alphas], index=alphas)
best_a = lasso_cv.idxmin()

lasso_cv.plot(logx=True, figsize=(9, 4), color="black", lw=2.5)
plt.axhline(subset_cv[best_subset], ls=":", color="gray", label="best of 128 subsets")
plt.axhline(cv_rmse(lambda n: ols_model(), X85, y85, folds), ls="--", color="gray", label="least squares, all 7")
plt.ylim(4, 5.6); plt.xlabel(r"$\alpha$ (log scale)"); plt.ylabel("CV RMSE ($1000s)"); plt.legend(fontsize=11)

print(f"best alpha = {best_a:.3g}: CV RMSE ${1000 * lasso_cv[best_a]:,.0f}, keeps {np.sum(np.abs(paths.loc[best_a]) > 1e-9)} of 7 features")

Too little penalty (left) overfits; too much (right) underfits. The curve is fairly flat from $\alpha = 10^{-2}$ to $10^{0}$, so, as last time, a simpler model is a reasonable choice. At $\alpha = 0.5$:

In [ ]:
a_simple = alphas[np.argmin(np.abs(alphas - 0.5))]
kept = X85.columns[np.abs(paths.loc[a_simple]) > 1e-9]
print(f"alpha = {a_simple:.2f}: keeps {list(kept)}, CV RMSE ${1000 * lasso_cv[a_simple]:,.0f}")

### Ridge on the same data

In [ ]:
ridge_cv = pd.Series([cv_rmse(lambda n: ridge_model(a, n), X85, y85, folds) for a in alphas], index=alphas)
best_r = ridge_cv.idxmin()
print(f"ridge: best alpha = {best_r:.3g}, CV RMSE ${1000 * ridge_cv[best_r]:,.0f} (all 7 features kept)")

In [ ]:
pd.DataFrame({
    "model": ["least squares, all 7", f"LASSO, alpha = {best_a:.2g} (best)", f"LASSO, alpha = {a_simple:.2g}",
              f"ridge, alpha = {best_r:.2g} (best)", "best of 128 subsets"],
    "features": [7, int(np.sum(np.abs(paths.loc[best_a]) > 1e-9)), len(kept), 7, len(best_subset)],
    "CV RMSE ($)": 1000 * np.array([cv_rmse(lambda n: ols_model(), X85, y85, folds), lasso_cv[best_a], lasso_cv[a_simple],
                                    ridge_cv[best_r], subset_cv[best_subset]]),
}).round(0)

Neither LASSO nor ridge reaches the best subset's CV RMSE. **But that number is optimistic:** it's the lowest of 128 CV scores on the same folds, so it's partly luck (Lesson 28's warning: comparing many models can overfit the CV error itself). Scored fairly on cars the search never saw, the best subset does about as well as ridge.

And beating an exhaustive search isn't the point: with 50 features that search would mean about $10^{15}$ models, while $\alpha$ is one number, whatever $p$ is.

### The sklearn shortcut: `LassoCV` and `RidgeCV`

`LassoCV` runs the whole CV loop for you. Two differences from our `cv_rmse`: it uses sklearn's scaling of $\alpha$ (so we pass $\alpha / 2$), and it doesn't standardize inside each fold, so we standardize the whole training set first. Its choice is close to ours:

In [ ]:
from sklearn.linear_model import LassoCV

X85_scaled = StandardScaler().fit_transform(X85)
cv_splits = [(np.where(folds != k)[0], np.where(folds == k)[0]) for k in sorted(folds.unique())]
lasso_cv_model = LassoCV(alphas=alphas / 2, cv=cv_splits, max_iter=100_000).fit(X85_scaled, y85)
print("LassoCV's choice, in our scaling: alpha =", round(2 * lasso_cv_model.alpha_, 4))
pd.Series(lasso_cv_model.coef_, index=X85.columns).round(2)   # $1000 per SD

`LassoCV` refits on all the training rows at the chosen $\alpha$. In a real project you'd then report the test error once. `RidgeCV` works the same way. **Always check** that the chosen $\alpha$ isn't at the edge of your grid; if it is, widen the grid and rerun.

## Summary

* **Regularization** adds a penalty on the size of the coefficients to the MSE (or, equivalently, limits them to a budget $Q$). Bigger $\alpha$ (smaller $Q$) means smaller coefficients and a simpler model; $\alpha = 0$ is least squares, and a huge $\alpha$ gives the flat line at $\theta_0$.
* **Correlated features** make many very different $\theta$s fit almost equally well. Least squares can land on huge, unstable ones; a penalty picks small ones.
* **LASSO (L1)** penalizes $\sum |\theta_j|$: its diamond has corners, so it sets some coefficients exactly to 0 and **selects features**. No closed form.
* **Ridge (L2)** penalizes $\sum \theta_j^2$: its circle has no corners, so it shrinks every coefficient but keeps all the features. Closed form $(\mathbb{Z}^T\mathbb{Z} + n\alpha I)^{-1}\mathbb{Z}^T\mathbb{Y}$.
* **Standardize** the features first (with the training rows' means and SDs), and don't penalize the intercept.
* **Choose $\alpha$ with cross-validation**, check that it isn't at the edge of the grid, and use the test set once, at the end.

---
# Optional practice: a market analysis with 35 features

The same tools on a bigger problem. This example uses sklearn's own scaling of `alpha` (no conversion), so its `alpha` values aren't comparable to the ones above, and it holds out a test set (there are over 1,000 rows).

# Example: A Market Analysis

A [market research project](https://doi.org/10.1080/02664763.2014.994480) for a pharmaceutical company wants to model consumer interest in purchasing a cold sore health-care product. They gather data from 1,023 consumers. Each consumer is asked to rate on a 10-point scale 35 factors according to whether the factor matters to them when considering purchasing a cold sore treatment. They also rate their interest in purchasing the product. 

Stan Lipovetsky and W. Michael Conklin (2015) Predictor relative importance
and matching regression parameters. _Journal of Applied Statistics_, 42:5, 1017-1031.

We begin by reading in the data:

In [ ]:
ma_df = pd.read_csv('data/market-analysis.csv')

ma_df.head()

The table below lists the 35 factors and provides their correlation to the outcome, their interest in purchasing the product. 




|  | Corr | Description |  | Corr | Description |
| --- | --- | --------- | --- | --- | --------- |
| x1  | 0.70 | provides soothing relief | x19 | 0.54 | has a non-messy application |
| x2  | 0.58 | moisturizes cold sore blister | x20 | 0.70 | good for any stage of a cold |
| x3  | 0.69 | provides long-lasting relief | x21 | 0.49 | easy to apply/take |
| x4  | 0.70 | provides fast-acting relief | x22 | 0.52 | package keeps from contamination |
| x5 | 0.72 | shortens duration of a cold | x23 | 0.57 | easy to dispense a right amount |
| x6  | 0.68 | stops the virus from spreading | x24 | 0.63 | worth the price it costs |
| x7 | 0.67| dries up cold sore | x25 | 0.57 | recommended most by pharamacists |
| x8 | 0.72 | heals fast | x26 | 0.54 | recommended by doctors |
| x9 | 0.72 | penetrates deep | x27 | 0.54 | FDA approved |
| x10 | 0.65 | relieves pain | x28 | 0.64 | a brand I trust |
| x11 |0.61 | prevents cold | x29 | 0.60 | clinically proven |
| x12 | 0.73 | prevents from getting worse | x30 | 0.68 | a brand I would recommend |
| x13 | 0.57 | medicated | x31 | 0.74 | an effective treatment |
| x14 | 0.61 | prescription strength | x32  |0.37 | portable |
| x15 | 0.63 | repairs damaged skin | x33 | 0.37 | discreet packaging |
| x16 | 0.67 | blocks virus from spreading | x34 | 0.55 | helps conceal cold sores |
| x17 | 0.42 | contains SPF | x35 | 0.63 | absorbs quickly |
| x18 | 0.57 | non-irritating | | | |

:::

In [ ]:
ma_df["y"].hist()

Based on their labels alone, some of these 35 features appear to measure similar aspects of desirability. We can compute the correlations between the explanatory variables to confirm this: 

In [ ]:
ma_df.corr().tail()

We see, for example, that the last feature, "absorbs quickly," is highly correlated with the first three: "provides soothing relief," "moisturizes," and "provides long-lasting relief."   

## Splitting into Training, Validation and Test Sets

Before we start to fit regularized models, we set up the design matrix and outcome vector and split the data into training, validation and test sets.  As usual we will only use the test set once (after selecting and fitting the model).

In [ ]:
Y = ma_df["y"]
X = ma_df.drop(columns=["y"])


In [ ]:
X_train_val, X_test, Y_train_val, Y_test = train_test_split(X, Y, 
                                                    test_size=0.2, 
                                                    random_state=100, 
                                                    shuffle=True)



# Further split train_val into training and validation (80/20)
X_train, X_val, Y_train, Y_val = train_test_split(X_train_val, Y_train_val, test_size=0.2, random_state=18)

print(f"Size of original set: {X.shape[0]} points")
print(f"Size of training set: {X_train.shape[0]} points")
print(f"Size of validation set: {X_val.shape[0]} points")
print(f"Size of test set: {X_test.shape[0]} points")

In [ ]:
import sklearn.linear_model as lm


In [ ]:
# Calculate RMSE:

def rmse(y, yhat):
    return np.sqrt(np.mean((y - yhat)**2))

## Using Regularization

### Converting Predictors to Standard Units



Before fitting a regularized model we need to get all of the predictors/features in standard units.

First, we'll standardize the training set, and then, when we go to evaluate the model on our validation set, we use the training set standardization on the validation-set. 

In [ ]:
X_train.head()

In [ ]:
X_train_centered = X_train- X_train.mean()
X_train_scaled = X_train_centered/X_train_centered.std()

X_val_scaled = (X_val - X_train.mean())/X_train_centered.std()

In [ ]:
X_train_scaled.std()

In [ ]:
# Option2 The [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) method helps us with this process:

#from sklearn.preprocessing import StandardScaler

#scalerX = StandardScaler().fit(X_train) 
#X_train_scaled = scalerX.transform(X_train)
#X_val_scaled = scalerX.transform(X_val)

We confirm that the means of the 35 features in the train set are all 0 and the SDs are all&nbsp;1: 

In [ ]:
np.allclose(X_train_scaled.mean(axis=0), 0)

In [ ]:
np.allclose(X_train_scaled.std(axis=0), 1)

It's important to note that this is not the case for the validation (or test sets), because we
use the averages and SDs from the training set to standardize the test set features:

In [ ]:
X_val_scaled.mean(axis=0)

## Lasso Regression 

To perform a lasso regression, we use the `Lasso` method in `scikit-learn`.  

In [ ]:
from sklearn.linear_model import Lasso

# Suppose we randomly chose alpha = 0.25 **We'll see more below on how to best choose alpha

# General Syntax for Fitting Lasso model with parameter alpha:

model_25 = Lasso(0.25)
model_25.fit(X_train_scaled, Y_train)



In [ ]:
model_25.coef_

Notice this method automatically selects the "best" features/predictors (for the given alpha) and sets the others to zero.  This is a way of feature selection.

In [ ]:
# Which features were selected?
X_train_val.loc[:,np.abs(model_25.coef_) > 0].columns

In [ ]:
# Using model to predict 
y_pred = model_25.predict(X_val_scaled)

rmse(y_pred, Y_val)

### Choosing Alpha:

### Some Visual Intuition

For demo/learning purposes only, let's see how the coefficients for the 35 features in the data frame change with $\alpha$. We set up a range of values for the regularization parameter and fit the lasso for each (this parameter is referred to as `alpha` in `Lasso`): 

In [ ]:


coefs = []
rmses = []
alphas = np.arange(0.01, 1.5, 0.01)

for a in alphas:
    model = Lasso(alpha=a)
    model.fit(X_train_scaled, Y_train)
    coefs.append(model.coef_)
    # We'll evaluate the validation RMSEs
    rmses.append(rmse(Y_val, model.predict(X_val_scaled)))

For each feature, we can overlay a line plot of the coefficient against $ \alpha $: 

In [ ]:

col_names = ["x" + str(v) for v in np.arange(1, 36, 1)]

coefs_df = pd.DataFrame(coefs, columns=col_names)

coefs_df["alpha"] = alphas
coefs_long = pd.melt(coefs_df, id_vars=["alpha"], value_vars=col_names)


fig = px.line(coefs_long, x="alpha", y="value", color="variable", log_x=True)
fig.update_layout(
    showlegend=False, width=550, height=250, yaxis_title="Coefficient",
    xaxis_title="alpha"
)



As $ \alpha $ increases, having more features in the model is penalized more heavily and many coefficients shrink to 0. The lefthand side of the plot shows greater model complexity and corresponds to small $ \alpha $. Notice that for $ \alpha = 0.5 $, nearly all of the coefficients are 0. 

We can also plot the RMSE as a function of $ \alpha $ to see how it changes with the increase in the penalty:

In [ ]:
px.line(x=alphas, y=rmses,
        labels={"x": "alpha", "y": "Validation RMSE"},
        width=350, height=250)

When $ \alpha $ reaches about 1.35, the penalty is so large that the lasso regression is simply fitting a constant model to the data, and the RMSE doesn't change. We are again faced with a model selection question, but this time it's in the form of deciding on $ \alpha $. 


## Choosing alpha:  

### Option 1: Based on desired number of predictors:

If you're interested in creating a simple model with just a select number of important predictors you can choose the alpha that results in your chosen number of predictors.  For example, suppose we want a model with just 6 predictors.    We can see in the alpha vs coefficient plot above, that an alpha value around .80 will result in a model with 6 predictors.

In [ ]:
model_80 = Lasso(alpha=.80)

model_80.fit(X_train_scaled, Y_train)



In [ ]:
model_80.coef_


In [ ]:
# Which features were selected?
X_train_val.loc[:,np.abs(model_80.coef_) > 0].columns

In [ ]:
rmse(Y_val, model_80.predict(X_val_scaled))

## Using Cross-Validation to Choose alpha:

If instead your goal is to find the "best" alpha that minimizes RMSE, we can use cross-validation to help us! 

We use the [`LassoCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html) method to perform 5-fold cross-validation to choose $\alpha$:

We'll use the full training_validation dataset (since we're not doing the validation manually anymore - it will be done 5 times as part of the 5-fold cross validation)
So first we need to re-standardize using the full training_val set:

In [ ]:
from sklearn.preprocessing import StandardScaler

scalerX_full = StandardScaler().fit(X_train_val) 
X_train_val_scaled = scalerX_full.transform(X_train_val)



In [ ]:
from sklearn.linear_model import LassoCV

#We'll choose alpha values between 0.01 and 1 based on our validation RMSE vs alpha plot above:)
lasso_cv_model = LassoCV(
    alphas=np.arange(0.01, 1, 0.01), cv=5)

In [ ]:
lasso_best = lasso_cv_model.fit(X_train_val_scaled, Y_train_val)

Cross-validation has chosen the following regularization parameter:

In [ ]:
lasso_best.alpha_

Let's find how many coefficients (out of the original 35) are not 0:

View which features are included in this Lasso Regression model:

In [ ]:
sum(np.abs(lasso_best.coef_) > 0)

In [ ]:
X_train_val.loc[:,np.abs(lasso_best.coef_) > 0].columns

## Run Final Version of Model on Test Data

In [ ]:
# Before running on the test set, we need to standardize the test data.
# We'll use the standardization we used on the training_val set:

X_test_scaled = scalerX_full.transform(X_test)

In [ ]:
# RMSE from test set (only run this once at end!)
Y_test_pred = lasso_best.predict(X_test_scaled)
rmse(Y_test, Y_test_pred)

### Peeking Under the Hood - Don't Try this at Home!

In [ ]:
# DON'T TRY THIS AT HOME!
# We'll peek under the hood and see if this actually performs better on the test set than
# the full OLS model using all the parameters.  

import sklearn.linear_model as lm

model_0 = lm.LinearRegression()

# Fit your linear model

model_0.fit(X_train_val, Y_train_val);



Y_test_pred_0 = model_0.predict(X_test)

In [ ]:
rmse(Y_test, Y_test_pred_0)


Our LASSO model uses only 18 of the 35 features, and its test RMSE (about 1.10) is lower than the test RMSE of OLS with all 35 features (about 1.15). Regularization helped here. (In practice we wouldn't look at the OLS model's test error; we're peeking only to illustrate.)

## L2 (Ridge Regression) 


`RidgeCV` works the same way as `LassoCV`: give it a grid of $\alpha$ values and it uses 5-fold cross-validation to pick the best one.

**Check the grid.** If the chosen $\alpha$ is the smallest or largest value in the grid, the best value may lie outside it, so widen the grid and try again. A grid on a log scale (here from 0.001 to 1,000) covers a wide range with few values.

In [ ]:
from sklearn.linear_model import RidgeCV

ridge_cv_model = RidgeCV(alphas=np.logspace(-3, 3, 61), cv=5)

In [ ]:
ridge_best = ridge_cv_model.fit(X_train_val_scaled, Y_train_val)

In [ ]:
ridge_best.alpha_

In [ ]:
# Is the chosen alpha at the edge of the grid?
print("Smallest and largest alpha in the grid:", ridge_cv_model.alphas.min(), ridge_cv_model.alphas.max())

Unlike LASSO, ridge regression shrinks the coefficients but doesn't set any of them exactly to 0:

In [ ]:
sum(np.abs(ridge_best.coef_) > 0)

In [ ]:
Y_test_pred = ridge_best.predict(X_test_scaled)
rmse(Y_test, Y_test_pred)

## Comparing the Models on the Test Set

(Again, for illustration only: in a real project we'd choose one model with cross-validation and look at its test error once.)

In [ ]:
comparison = pd.DataFrame({
    "Model": ["OLS (all 35 features)", f"LASSO (alpha = {lasso_best.alpha_:.2f})", f"Ridge (alpha = {ridge_best.alpha_:.3g})"],
    "Nonzero coefficients": [35, int(sum(np.abs(lasso_best.coef_) > 0)), int(sum(np.abs(ridge_best.coef_) > 0))],
    "Test RMSE": [rmse(Y_test, model_0.predict(X_test)),
                  rmse(Y_test, lasso_best.predict(X_test_scaled)),
                  rmse(Y_test, ridge_best.predict(X_test_scaled))],
})
comparison.round(3)



Sometimes we prefer one type of regularization over the other because it maps more closely to the domain we are working with. 

Some outcomes result from a few highly influential features. We prefer lasso regularization in these situations because it discards unneeded features. Both $L_2$ and $L_1$ serve as useful throttles to navigate between under- and overfitting.


For example, if we know that the phenomenon we are trying to model results from many small factors, we might prefer ridge regression because it won't discard these factors. 


## Other types of Regularization  (Out of Scope):

Elastic Net Regularization is a Combination of Both Methods. [You can read more about it here:](https://www.geeksforgeeks.org/what-is-elasticnet-in-sklearn/) 
